# 🎯 unilabellm — YOLO Training Notebook

Bu notebook **unilabellm** ile dışa aktarılan birleşik YOLO datasetini Kaggle GPU'sunda eğitmek için hazırlanmıştır.

**Beklenen dataset yapısı** (unilabellm export çıktısı, zip halinde yükleyin):
```
/kaggle/input/{dataset-slug}/
├── data.yaml
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

| Ayar | Değer |
|------|-------|
| Model | YOLO11s (değiştirilebilir) |
| Epoch | 100 |
| Image size | 640 |
| Batch | 16 |


## 1 · Kurulum & GPU kontrolü

In [ ]:
!pip install -q ultralytics

import os, shutil, yaml
from pathlib import Path
from collections import Counter
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2 · Dataset tespiti & data.yaml güncelleme

unilabellm'in data.yaml'ı **mutlak path** yerine **görece path** içerir. Kaggle'da çalışması için path'i güncelliyoruz.

In [ ]:
# ─── Kaggle input klasöründe data.yaml bul ───────────────────────────────────
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR   = Path("/kaggle/working")

data_yaml_candidates = sorted(INPUT_ROOT.rglob("data.yaml"))
if not data_yaml_candidates:
    raise FileNotFoundError(
        "data.yaml bulunamadı! Lütfen unilabellm export ZIP'ini "
        "Kaggle dataseti olarak ekleyin."
    )

# Birden fazla varsa ilkini al (tek dataset eklendiyse sorun yok)
SRC_YAML = data_yaml_candidates[0]
DATASET_DIR = SRC_YAML.parent
print("Dataset dizini:", DATASET_DIR)
print("data.yaml bulundu:", SRC_YAML)

# ─── data.yaml'ı /kaggle/working'e kopyala ve path'i güncelle ────────────────
WORK_YAML = WORK_DIR / "data.yaml"

with open(SRC_YAML) as f:
    cfg = yaml.safe_load(f)

# Kaggle'da path mutlak olmalı
cfg["path"] = str(DATASET_DIR)
cfg["train"] = "images/train"
cfg["val"]   = "images/val"
cfg["test"]  = "images/test"

# names: list veya dict olabilir — normalize et
if isinstance(cfg.get("names"), list):
    names_list = cfg["names"]
elif isinstance(cfg.get("names"), dict):
    names_list = [cfg["names"][k] for k in sorted(cfg["names"])]
    cfg["names"] = names_list
else:
    raise ValueError("data.yaml'da 'names' alanı bulunamadı")

with open(WORK_YAML, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print("\n=== data.yaml (güncellenmiş) ===")
print(WORK_YAML.read_text())

## 3 · Dataset istatistikleri

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def count_split(split):
    img_dir = DATASET_DIR / "images" / split
    lbl_dir = DATASET_DIR / "labels" / split
    if not img_dir.exists():
        return 0, Counter()
    imgs  = len(list(img_dir.glob("*")))
    counts = Counter()
    for txt in lbl_dir.glob("*.txt"):
        for line in txt.read_text().strip().splitlines():
            parts = line.split()
            if parts:
                counts[int(parts[0])] += 1
    return imgs, counts

total_imgs = 0
total_boxes = Counter()
print(f"{'Split':<8} {'Images':>8} {'Boxes':>8}")
print("-" * 28)
for split in ["train", "val", "test"]:
    imgs, counts = count_split(split)
    boxes = sum(counts.values())
    total_imgs  += imgs
    total_boxes += counts
    print(f"{split:<8} {imgs:>8,} {boxes:>8,}")
print("-" * 28)
print(f"{'TOTAL':<8} {total_imgs:>8,} {sum(total_boxes.values()):>8,}")

# Sınıf dağılımı grafiği
class_counts = {names_list[k]: v for k, v in sorted(total_boxes.items()) if k < len(names_list)}
if class_counts:
    fig, ax = plt.subplots(figsize=(max(6, len(class_counts)*1.2), 4))
    bars = ax.bar(class_counts.keys(), class_counts.values(), color="#0070f3", edgecolor="#003d8f")
    ax.set_title("Sınıf başına annotation sayısı", fontsize=13, fontweight="bold")
    ax.set_ylabel("Annotation sayısı")
    ax.tick_params(axis="x", rotation=30)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f"{bar.get_height():,}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
    print("\nSınıflar:", names_list)

## 4 · Model seçimi

| Model | Parametre | Hız | Doğruluk | Önerildiği durum |
|-------|-----------|-----|----------|-------------------|
| `yolo11n.pt` | 2.6M | ⚡⚡⚡ | ★★☆ | Hızlı deneme |
| `yolo11s.pt` | 9.4M | ⚡⚡ | ★★★ | **Denge (önerilen)** |
| `yolo11m.pt` | 20M | ⚡ | ★★★★ | Daha iyi doğruluk |
| `yolo11x.pt` | 56M | 🐢 | ★★★★★ | Maksimum doğruluk |


In [ ]:
# ─── Buradan modeli seç ───────────────────────────────────────────────────────
MODEL_NAME = "yolo11s.pt"   # yolo11n / yolo11s / yolo11m / yolo11x
# ─────────────────────────────────────────────────────────────────────────────

from ultralytics import YOLO

# Kaggle'da /kaggle/input içinde yüklü model var mı kontrol et
uploaded_models = list(INPUT_ROOT.rglob(MODEL_NAME))
if uploaded_models:
    model_path = str(uploaded_models[0])
    print(f"Mevcut model kullanılıyor: {model_path}")
else:
    model_path = MODEL_NAME   # ultralytics otomatik indirir
    print(f"Model internet üzerinden indirilecek: {MODEL_NAME}")

model = YOLO(model_path)
print(f"Model yüklendi: {model_path}")
print(f"Sınıf sayısı dataset'te: {cfg['nc']}")

## 5 · Eğitim

> **Batch ayarı:** GPU VRAM'ınıza göre `batch` değerini ayarlayın.
> - T4 (16GB) → 16
> - P100 (16GB) → 16  
> - Tesla V100 (32GB) → 32

In [ ]:
# ─── Eğitim hiperparametreleri ────────────────────────────────────────────────
EPOCHS   = 100
IMGSZ    = 640
BATCH    = 16     # OOM olursa 8 yap
PATIENCE = 15     # Early stopping
RUN_NAME = "unilabellm_run"
# ─────────────────────────────────────────────────────────────────────────────

results = model.train(
    data      = str(WORK_YAML),
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    batch     = BATCH,
    device    = 0 if torch.cuda.is_available() else "cpu",
    workers   = 2,
    project   = str(WORK_DIR),
    name      = RUN_NAME,
    pretrained= True,
    patience  = PATIENCE,
    cache     = False,
    save      = True,
    plots     = True,
    verbose   = True,
    # Askeri dataset için hafif augmentation düzeltmeleri
    degrees   = 10,    # Rotasyon açısı (kuşbakışı drone görüntüleri için)
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    mosaic    = 1.0,
)

RUN_DIR = WORK_DIR / RUN_NAME
print("\nEğitim tamamlandı!")
print("Çıktı dizini:", RUN_DIR)

## 6 · Sonuçlar & Metrikler

In [ ]:
from IPython.display import Image as IPImage, display
import matplotlib.pyplot as plt

# Eğitim grafiklerini göster
for plot_name in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        print(f"\n--- {plot_name} ---")
        display(IPImage(str(plot_path), width=900))

# En iyi ağırlık
best_pt = RUN_DIR / "weights" / "best.pt"
print("\nEn iyi model:", best_pt)
print("Mevcut mu:", best_pt.exists())

## 7 · Doğrulama (Validation)

In [ ]:
best_model = YOLO(str(best_pt))

val_results = best_model.val(
    data    = str(WORK_YAML),
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = 0 if torch.cuda.is_available() else "cpu",
    split   = "val",
    plots   = True,
    verbose = True,
)

print("\n=== Validation Özeti ===")
print(f"mAP@0.5     : {val_results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"Precision   : {val_results.box.mp:.4f}")
print(f"Recall      : {val_results.box.mr:.4f}")

# Sınıf bazlı metrikler
print("\n=== Sınıf Bazlı mAP@0.5 ===")
for i, name in enumerate(names_list):
    try:
        ap = val_results.box.ap50[i]
        print(f"  {name:<30} {ap:.4f}")
    except Exception:
        pass

## 8 · Test görüntülerinde tahmin örneği

In [ ]:
import random
from IPython.display import Image as IPImage, display

test_images = list((DATASET_DIR / "images" / "test").glob("*"))
if not test_images:
    test_images = list((DATASET_DIR / "images" / "val").glob("*"))

if test_images:
    sample_imgs = random.sample(test_images, min(6, len(test_images)))

    pred_results = best_model.predict(
        source    = sample_imgs,
        imgsz     = IMGSZ,
        conf      = 0.25,
        save      = True,
        project   = str(WORK_DIR),
        name      = "predictions",
        device    = 0 if torch.cuda.is_available() else "cpu",
    )

    pred_dir = WORK_DIR / "predictions"
    saved = sorted(pred_dir.glob("*.jpg")) + sorted(pred_dir.glob("*.png"))
    for p in saved[:6]:
        display(IPImage(str(p), width=640))
else:
    print("Test görseli bulunamadı.")

## 9 · Model dışa aktarma

In [ ]:
# ─── İstediğin formatı seç ───────────────────────────────────────────────────
EXPORT_FORMAT = "onnx"   # onnx | torchscript | tflite | coreml | engine(TensorRT)
# ─────────────────────────────────────────────────────────────────────────────

exported_path = best_model.export(
    format  = EXPORT_FORMAT,
    imgsz   = IMGSZ,
    device  = 0 if torch.cuda.is_available() else "cpu",
    dynamic = True,      # ONNX için dinamik batch
    simplify= True,      # ONNX simplify
)
print("Dışa aktarıldı:", exported_path)

## 10 · Sonuçları paketle (Kaggle output)

In [ ]:
import zipfile
from pathlib import Path

OUT_ZIP = WORK_DIR / f"{RUN_NAME}_output.zip"

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    # Ağırlıklar
    for pt in (RUN_DIR / "weights").glob("*.pt"):
        zf.write(pt, f"weights/{pt.name}")

    # Metrik grafikleri
    for plot in RUN_DIR.glob("*.png"):
        zf.write(plot, f"plots/{plot.name}")

    # results.csv
    results_csv = RUN_DIR / "results.csv"
    if results_csv.exists():
        zf.write(results_csv, "results.csv")

    # Kullanılan data.yaml
    zf.write(WORK_YAML, "data.yaml")

    # Export edilen model (onnx vs)
    if exported_path and Path(exported_path).exists():
        ep = Path(exported_path)
        zf.write(ep, f"exported/{ep.name}")

size_mb = round(OUT_ZIP.stat().st_size / 1_048_576, 1)
print(f"✅ Paket hazır: {OUT_ZIP}")
print(f"   Boyut: {size_mb} MB")
print("\n/kaggle/working klasöründen 'Output' sekmesine giderek indirebilirsiniz.")